# SFT Penilai Kelogisan Unsur TOR (per butir) — **Phi-4-mini-instruct** — **Kaggle T4**

Adaptasi dari `SFT_TOR_PerButir_Qwen3_4B_Kaggle.ipynb` untuk:
1. **Model `microsoft/Phi-4-mini-instruct`** (3.8B, lebih ringan dari Qwen3-4B) — pembanding.
2. **Dataset `data neo`** (format per-butir: 1 baris menilai 1–3 butir, `prompt`/`completion` + `_meta`).
3. **Validasi ke TOR ASLI**: split *test* 100% dokumen asli (57 TOR, 928 baris), **tanpa kebocoran** ke train/val
   (dipisah per-dokumen). Angka test di sini merepresentasikan performa pada TOR nyata yang berantakan,
   bukan data sintetis segenerasi dengan train.

**Yang dihasilkan**: metrik per-butir (P/R/F1 kelas *tidak logis*), akurasi level dokumen (LOLOS/REVISI),
per-metode kerusakan, perplexity, dan BERTScore — untuk **Phi hasil fine-tune** dan **Phi mentah (baseline)**.

> T4 catatan: tidak ada bf16 → `fp16`; tidak ada flash-attn (butuh Ampere+) → `sdpa`/`eager`.
> Jalankan sel **berurutan**. Smoke test (sel 16) wajib sekali sebelum training penuh.


## 1. Cek GPU & instal library

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
!pip install -q -U "transformers>=4.51" "trl>=0.19" "peft>=0.15" accelerate datasets liger-kernel
!pip uninstall -y -q torchao 2>/dev/null || echo "torchao tidak terpasang"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [3]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # pakai 1 GPU T4 saja (hindari DataParallel otomatis)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import json, gc, math, re, time, inspect, random
from collections import Counter, defaultdict
import torch, transformers, trl, peft
import pandas as pd
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "GPU tidak terdeteksi - aktifkan Accelerator GPU T4 di Settings"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
BF16_OK = torch.cuda.is_bf16_supported()
print("bf16 didukung GPU ini:", BF16_OK, "(T4 = False -> training pakai fp16)")
try:
    import liger_kernel  # noqa: F401
    HAS_LIGER = True
except Exception:
    HAS_LIGER = False
print("liger-kernel:", HAS_LIGER)

torch 2.11.0+cu130 | transformers 5.23.0 | trl 1.18.1 | peft 0.21.3
GPU: Tesla T4 16 GB
bf16 didukung GPU ini: True (T4 = False -> training pakai fp16)
liger-kernel: True


## 2. Konfigurasi
Ubah path & hyperparameter di sini saja.

In [4]:
# === Path Kaggle ===
DATA_DIR = "/kaggle/input/datasets/mikailachmad/data-neo/data neo"  # <- path dataset Anda (konfirmasi dari !ls)
OUT_DIR  = "/kaggle/working/phi4mini-perbutir-lora"
os.makedirs(OUT_DIR, exist_ok=True)

SKIP_TRAINING = True
ADAPTER_SRC_DIR = "/kaggle/input/phi4mini-final-adapter"
ADAPTER_DIR = ADAPTER_SRC_DIR if SKIP_TRAINING else f"{OUT_DIR}/final_adapter"

# (opsional) HF token dari Kaggle Secrets -> unduh lebih cepat / hindari rate limit
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _tok
    from huggingface_hub import login; login(_tok)
    print("HF token terpasang")
except Exception as e:
    print("HF token tidak diset (opsional, model publik tetap bisa diunduh):", str(e)[:80])

MODEL_ID = "microsoft/Phi-4-mini-instruct"
MAX_LEN = 2048            # p95 data ~1368 token; 2048 cukup, lebih cepat & hemat memori di T4
EPOCHS = 3
LR = 2e-4
BATCH = 2                  # T4 16GB: kecilkan dari 4 -> 2
GRAD_ACCUM = 8             # batch efektif tetap 16
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
VAL_SUBSET = 400
EVAL_EVERY = 100
SEED = 42
MAX_NEW_TOKENS = 512       # jawaban multi-butir (unit M/O berisi 3 butir) -> lebih panjang
GEN_BATCH = 8              # batch inferensi evaluasi (T4)
USE_LIGER = False         # Phi-4-mini arsitektur beda dari Qwen; liger-kernel bisa tak cocok -> matikan
TRUST_REMOTE_CODE = False # Phi-4 sudah native di transformers>=4.49; loader auto-coba True bila perlu
ATTN_IMPL = "sdpa"        # T4 tak dukung flash-attn; loader auto-fallback ke "eager" bila error

for f in ("train", "val", "test", "train_meta", "val_meta", "test_meta"):
    p = f"{DATA_DIR}/{f}.jsonl"
    print(f"{f}.jsonl", "OK" if os.path.exists(p) else "TIDAK ADA -> cek DATA_DIR")

HF token tidak diset (opsional, model publik tetap bisa diunduh): Connection error trying to communicate with service.
train.jsonl OK
val.jsonl OK
test.jsonl OK
train_meta.jsonl OK
val_meta.jsonl OK
test_meta.jsonl OK


## 3. Muat data (format multi-butir `kelompok_out`)
Tiap baris berformat *prompt-completion* (system + user -> jawaban JSON berkunci kode butir). Satu baris bisa menilai **beberapa butir** sekaligus (unit M: I.M.2/I.M.3/I.M.1; unit O: I.O.2/I.O.3/I.O.1). Loss hanya dihitung di jawaban.

In [5]:
from datasets import Dataset

BUTIR_NAME = {
    "I.M.2": "Dasar Hukum", "I.M.3": "Gambaran Umum", "I.M.1": "Latar Belakang",
    "I.O.2": "Metode Pelaksanaan", "I.O.3": "Tahapan dan Waktu Pelaksanaan",
    "I.O.1": "Strategi Pencapaian Keluaran", "I.N.1": "Penerima Manfaat",
    "I.P": "Kurun Waktu Pencapaian Keluaran", "I.Q": "Biaya yang Diperlukan",
    "I.R": "Manajemen Risiko",
}
BUTIR_ORDER = ["I.M.2", "I.M.3", "I.M.1", "I.O.2", "I.O.3", "I.O.1", "I.N.1", "I.P", "I.Q", "I.R"]

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

rows = {s: load_jsonl(f"{DATA_DIR}/{s}.jsonl") for s in ("train", "val", "test")}
meta = {s: load_jsonl(f"{DATA_DIR}/{s}_meta.jsonl") for s in ("train", "val", "test")}
for s in rows:
    assert len(rows[s]) == len(meta[s]), f"{s}: jumlah baris data dan meta berbeda"
    for r, m in zip(rows[s], meta[s]):
        r["_meta"] = m
    n_butir = sum(len(m["labels"]) for m in meta[s])
    n_tidak = sum(1 for m in meta[s] for v in m["labels"].values() if not v)
    print(f"{s:5s} {len(rows[s]):5d} baris | {n_butir} butir dinilai | logis {n_butir-n_tidak} / tidak logis {n_tidak}")
GROUPS = sorted({m["group"] for m in meta["train"]})
print("group/unit:", GROUPS)
print("contoh jawaban:", rows["train"][0]["completion"][0]["content"][:300])

train  3899 baris | 7523 butir dinilai | logis 4529 / tidak logis 2994
val    2615 baris | 4277 butir dinilai | logis 3627 / tidak logis 650
test    928 baris | 1216 butir dinilai | logis 810 / tidak logis 406
group/unit: ['Biaya yang Diperlukan', 'Kurun Waktu Pencapaian Keluaran', 'Latar Belakang', 'Manajemen Risiko', 'Penerima Manfaat', 'Strategi Pencapaian Keluaran']
contoh jawaban: {"I.Q": {"logis": true, "alasan": "Bagian ini mencantumkan nilai anggaran rupiah yang jelas untuk pelaksanaan kegiatan evaluasi, yaitu 'sebesar Rp1.850.000.000,- (satu miliar delapan ratus lima puluh juta rupiah)'.", "saran": ""}}


## 4. Tokenizer & cek panjang token
Baris yang lebih panjang dari `MAX_LEN` dibuang dari train/val (tidak dipotong) dan dilaporkan.

In [6]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Token henti untuk generate. Phi-4-mini mengakhiri giliran dengan <|end|>;
# sertakan juga eos bawaan agar greedy decoding berhenti tepat waktu.
_eos = {tok.eos_token_id}
for _t in ("<|end|>", "<|endoftext|>", "<|im_end|>"):
    try:
        _i = tok.convert_tokens_to_ids(_t)
        if isinstance(_i, int) and _i >= 0 and _i != tok.unk_token_id:
            _eos.add(_i)
    except Exception:
        pass
EOS_IDS = sorted(x for x in _eos if x is not None)
print("EOS_IDS:", EOS_IDS)

def n_tokens(r):
    text = tok.apply_chat_template(r["prompt"] + r["completion"], tokenize=False)
    return len(tok(text, add_special_tokens=False)["input_ids"])

for s in ("train", "val", "test"):
    for r in rows[s]:
        r["_len"] = n_tokens(r)
    L = sorted(r["_len"] for r in rows[s])
    print(f"{s:5s} median {L[len(L)//2]:5d}  p95 {L[int(len(L)*.95)]:5d}  maks {L[-1]:5d}  > MAX_LEN: {sum(x > MAX_LEN for x in L)}")

for s in ("train", "val"):
    n0 = len(rows[s])
    rows[s] = [r for r in rows[s] if r["_len"] <= MAX_LEN]
    if len(rows[s]) < n0:
        print(f"PERINGATAN: {n0-len(rows[s])} baris {s} > MAX_LEN dibuang")

def to_ds(rs):
    return Dataset.from_list([{"prompt": r["prompt"], "completion": r["completion"]} for r in rs])

# val kecil berimbang antar-unit untuk eval selama training
rng = random.Random(SEED)
by = defaultdict(list)
for r in rows["val"]:
    by[r["_meta"]["group"]].append(r)
per = max(1, VAL_SUBSET // len(by))
val_small = [x for k in sorted(by) for x in rng.sample(by[k], min(per, len(by[k])))]
train_ds, val_ds = to_ds(rows["train"]), to_ds(val_small)
print(f"train {len(train_ds)} baris | val (eval selama training) {len(val_ds)} baris")

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

EOS_IDS: [199999, 200020]
train median   725  p95  1178  maks  4638  > MAX_LEN: 21
val   median   570  p95  1053  maks  1332  > MAX_LEN: 0
test  median   731  p95  1760  maks  3763  > MAX_LEN: 30
PERINGATAN: 21 baris train > MAX_LEN dibuang
train 3878 baris | val (eval selama training) 396 baris


## 5. Model & LoRA (fp16)

In [7]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, PeftModel

def _from_pretrained(**kw):
    # transformers baru pakai `dtype=`, lama pakai `torch_dtype=`
    try:
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, **kw)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, **kw)

def load_base():
    # T4: fp16 (tak ada bf16) + sdpa/eager (tak ada flash-attn).
    # trust_remote_code=True TIDAK dicoba: Phi-4-mini sudah native di transformers>=4.49,
    # dan kode custom yang didownload dari HF hub (modeling_phi3.py) terbukti tidak kompatibel
    # dengan transformers versi baru (ImportError: LossKwargs hilang di transformers 5.x).
    attempts = [
        dict(attn_implementation=ATTN_IMPL, trust_remote_code=False, device_map={"": 0}),
        dict(attn_implementation="eager",  trust_remote_code=False, device_map={"": 0}),
    ]
    last = None
    for i, kw in enumerate(attempts, 1):
        try:
            m = _from_pretrained(**kw)
            print(f"model dimuat (percobaan {i}): attn={kw['attn_implementation']}, "
                  f"trust_remote_code={kw['trust_remote_code']}")
            m.config.use_cache = False
            return m
        except Exception as e:
            last = e
            print(f"percobaan {i} gagal (attn={kw['attn_implementation']}, "
                  f"trc={kw['trust_remote_code']}): {str(e)[:160]}")
    raise last

lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                      target_modules="all-linear", task_type="CAUSAL_LM", bias="none")


## 6. Konfigurasi trainer
Nama parameter TRL berbeda antar versi; sel ini menyesuaikan otomatis. **fp16** dipakai (bukan bf16) karena T4.

In [8]:
from trl import SFTConfig, SFTTrainer
from transformers import TrainerCallback

STEPS_PER_EPOCH = math.ceil(len(train_ds) / (BATCH * GRAD_ACCUM))
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS
print("langkah per epoch:", STEPS_PER_EPOCH, "| total:", TOTAL_STEPS)

def make_config(**over):
    want = dict(
        output_dir=OUT_DIR, num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR,
        lr_scheduler_type="cosine", warmup_ratio=0.05, weight_decay=0.0,
        fp16=True, bf16=False,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=10, eval_strategy="steps", eval_steps=EVAL_EVERY,
        save_strategy="steps", save_steps=EVAL_EVERY, save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        group_by_length=True,
        report_to="none", seed=SEED, completion_only_loss=True, packing=False,
        use_liger_kernel=USE_LIGER, dataset_num_proc=2,
    )
    want.update(over)
    params = set(inspect.signature(SFTConfig.__init__).parameters)
    want["max_length" if "max_length" in params else "max_seq_length"] = MAX_LEN
    if "warmup_ratio" not in params:
        steps = want["max_steps"] if want.get("max_steps", -1) > 0 else TOTAL_STEPS
        want["warmup_steps"] = max(1, round(0.05 * steps))
        want.pop("warmup_ratio", None)
    if "eval_strategy" not in params and "evaluation_strategy" in params:
        want["evaluation_strategy"] = want.pop("eval_strategy")
    dropped = [k for k in want if k not in params]
    if dropped:
        print("parameter tidak didukung versi ini (diabaikan):", dropped)
    return SFTConfig(**{k: v for k, v in want.items() if k in params})

class MemCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if state.global_step in (1, 5) or state.global_step % 50 == 0:
            print(f"[step {state.global_step}/{state.max_steps}] puncak memori GPU {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

def make_trainer(model, cfg, train, val=None):
    kw = dict(model=model, args=cfg, train_dataset=train, eval_dataset=val,
              peft_config=lora_cfg, callbacks=[MemCallback()])
    sig = set(inspect.signature(SFTTrainer.__init__).parameters)
    kw["processing_class" if "processing_class" in sig else "tokenizer"] = tok
    return SFTTrainer(**kw)

langkah per epoch: 243 | total: 729


## 7. Smoke test (WAJIB sekali)
Melatih 3 langkah memakai baris-baris **terpanjang** (paling berat). Kalau lolos tanpa *out of memory*, training penuh pasti muat. Sel ini juga memperkirakan lama training.
Kalau OOM: turunkan `BATCH` ke 1 dan naikkan `GRAD_ACCUM` ke 16, lalu ulangi sel 2 ke bawah.

In [9]:
if not SKIP_TRAINING:
    torch.cuda.reset_peak_memory_stats()
    model = load_base()
    longest = sorted(rows["train"], key=lambda r: -r["_len"])[: BATCH * 3]
    smoke_cfg = make_config(output_dir="/kaggle/working/smoke", max_steps=3, gradient_accumulation_steps=1,
                            eval_strategy="no", save_strategy="no", load_best_model_at_end=False,
                            logging_steps=1, group_by_length=False)
    smoke = make_trainer(model, smoke_cfg, to_ds(longest))
    t0 = time.time(); smoke.train(); dt = (time.time() - t0) / 3
    mean_len = sum(r["_len"] for r in rows["train"]) / len(rows["train"])
    est = dt * (mean_len / longest[0]["_len"]) * GRAD_ACCUM * TOTAL_STEPS / 60
    print(f"SMOKE TEST OK - puncak memori {torch.cuda.max_memory_allocated()/1e9:.1f} GB, "
          f"{dt:.1f} detik/langkah untuk batch terpanjang (~{longest[0]['_len']} token)")
    print(f"Perkiraan kasar training penuh: ~{est:.0f} menit (+ eval berkala)")
    b = smoke.train_dataset[0]
    if "completion_mask" in b:
        print("token yang dihitung loss:", sum(b["completion_mask"]), "dari", len(b["input_ids"]))
    del smoke, model; gc.collect(); torch.cuda.empty_cache()
else:
    print("SKIP_TRAINING=True -> smoke test dilewati")

SKIP_TRAINING=True -> smoke test dilewati


## 8. Training penuh
Checkpoint tiap `EVAL_EVERY` langkah disimpan ke `/kaggle/working`. Kalau sesi terputus (batas Kaggle ~9-12 jam): jalankan ulang sel 1-6 lalu sel ini, training **dilanjutkan** dari checkpoint terakhir.

In [10]:
if not SKIP_TRAINING:
    torch.cuda.reset_peak_memory_stats()
    model = load_base()
    trainer = make_trainer(model, make_config(), train_ds, val_ds)
    has_ckpt = any(d.startswith("checkpoint-") for d in os.listdir(OUT_DIR))
    t0 = time.time()
    trainer.train(resume_from_checkpoint=True if has_ckpt else None)
    print(f"selesai dalam {(time.time()-t0)/60:.0f} menit; puncak memori {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

    trainer.model.save_pretrained(ADAPTER_DIR)
    tok.save_pretrained(ADAPTER_DIR)
    print("adapter (checkpoint terbaik menurut eval_loss) disimpan ke", ADAPTER_DIR)
else:
    assert ADAPTER_SRC_DIR and os.path.isdir(ADAPTER_SRC_DIR), \
        "SKIP_TRAINING=True tapi ADAPTER_SRC_DIR kosong/tidak ada -> isi path adapter di sel 2 (Konfigurasi)"
    print(f"SKIP_TRAINING=True -> training dilewati, pakai adapter dari {ADAPTER_DIR}")

SKIP_TRAINING=True -> training dilewati, pakai adapter dari /kaggle/input/phi4mini-final-adapter


In [11]:
if not SKIP_TRAINING:
    hist = pd.DataFrame(trainer.state.log_history)
    cols = [c for c in ("step", "epoch", "loss", "eval_loss") if c in hist]
    display(hist[cols].dropna(how="all", subset=[c for c in ("loss", "eval_loss") if c in hist]).tail(40))
else:
    print("SKIP_TRAINING=True -> tidak ada log training baru untuk ditampilkan")

SKIP_TRAINING=True -> tidak ada log training baru untuk ditampilkan


## 9. Evaluasi di test set (per butir)
Inferensi greedy secara **batch**. Setiap baris bisa berisi beberapa butir; metrik dihitung **per butir** (10 butir, termasuk induk Latar Belakang & Strategi). Metrik:
- **per butir**: akurasi, *balanced accuracy*, precision / recall / F1 untuk kelas **tidak logis**,
- **per metode kerusakan**: proporsi benar tiap jenis (ASLI/INDUK = unsur utuh; HAPUS_*, TUKAR_*, KOSONG, SIMBOL = kerusakan),
- **per dokumen** (TOR): akurasi LOLOS/REVISI (REVISI bila ada >=1 butir diprediksi tidak logis),
- jumlah jawaban/butir yang **tak terbaca** (dianggap `logis: true`, kasus terburuk untuk recall).

Set `EVAL_BASELINE=True` untuk mengukur model dasar tanpa fine-tune sebagai pembanding.

In [12]:
EVAL_BASELINE = True   # bandingkan Phi hasil fine-tune vs Phi mentah di test asli

def parse_output(txt):
    t = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    cands = [t]
    m = re.search(r"\{.*\}", t, re.S)
    if m:
        cands.append(m.group(0))
    for cand in cands:
        try:
            j = json.loads(cand)
            if isinstance(j, dict):
                return j
        except Exception:
            pass
    return None

def get_logis(pred_j, code):
    if isinstance(pred_j, dict):
        v = pred_j.get(code)
        if isinstance(v, dict) and isinstance(v.get("logis"), bool):
            return v["logis"]
    return None

@torch.no_grad()
def predict(model, rs, label="", bs=GEN_BATCH):
    model.eval(); model.config.use_cache = True
    try: model.gradient_checkpointing_disable()
    except Exception: pass
    tok.padding_side = "left"
    order = sorted(range(len(rs)), key=lambda i: -rs[i]["_len"])
    out = [None] * len(rs)
    t0 = time.time()
    for k in range(0, len(order), bs):
        idx = order[k:k + bs]
        texts = [tok.apply_chat_template(rs[i]["prompt"], tokenize=False, add_generation_prompt=True) for i in idx]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS)
        for j, i in enumerate(idx):
            txt = tok.decode(gen[j, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            out[i] = {"raw": txt, "pred": parse_output(txt)}
        done = k + len(idx)
        if (k // bs) % 10 == 0 or done == len(rs):
            el = time.time() - t0
            print(f"{label} {done}/{len(rs)}  {el/60:.1f} menit, sisa ~{el/done*(len(rs)-done)/60:.1f} menit")
    tok.padding_side = "right"
    return out

def prf(tp, fp, fn, tn):
    p = tp / (tp + fp) if tp + fp else float("nan")
    r = tp / (tp + fn) if tp + fn else float("nan")
    f = 2 * p * r / (p + r) if p == p and r == r and p + r else float("nan")
    spec = tn / (tn + fp) if tn + fp else float("nan")
    return p, r, f, spec

def evaluate(rs, preds, name):
    bad_rows = sum(p["pred"] is None for p in preds)
    missing = 0; total = 0
    per = defaultdict(Counter); meth = defaultdict(Counter); docs = defaultdict(lambda: [None, False])
    for r, pdct in zip(rs, preds):
        m = r["_meta"]; pred_j = pdct["pred"]
        d = docs[m["doc_id"]]; d[0] = m["doc_status"]
        for code, gold in m["labels"].items():
            total += 1
            pl = get_logis(pred_j, code)
            if pl is None:
                pl = True; missing += 1          # butir tak terbaca -> dianggap logis (kasus terburuk utk recall)
            u = BUTIR_NAME.get(code, code)
            gneg, pneg = (not gold), (not pl)
            key = "tp" if gneg and pneg else "fn" if gneg else "fp" if pneg else "tn"
            per[u][key] += 1; per["SEMUA"][key] += 1
            meth[m["metode"][code]]["benar" if pl == gold else "salah"] += 1
            d[1] = d[1] or pneg
    tab = []
    for u in [BUTIR_NAME[c] for c in BUTIR_ORDER] + ["SEMUA"]:
        c = per[u]
        n = sum(c.values())
        if n == 0:
            continue
        p, r, f, spec = prf(c["tp"], c["fp"], c["fn"], c["tn"])
        tab.append({"butir": u, "n": n, "tidak_logis_gold": c["tp"] + c["fn"], "akurasi": (c["tp"] + c["tn"]) / n,
                    "balanced_acc": (r + spec) / 2, "precision_TIDAK": p, "recall_TIDAK": r, "F1_TIDAK": f})
    df = pd.DataFrame(tab)
    dm = pd.DataFrame([{"metode": k, "n": sum(v.values()), "benar": v["benar"] / sum(v.values())}
                       for k, v in sorted(meth.items())])
    dc = Counter()
    for status, prev in docs.values():
        dc[(status, "REVISI" if prev else "LOLOS")] += 1
    ndoc = len(docs); okdoc = dc[("LOLOS", "LOLOS")] + dc[("REVISI", "REVISI")]
    tpd, fpd, fnd = dc[("REVISI", "REVISI")], dc[("LOLOS", "REVISI")], dc[("REVISI", "LOLOS")]
    pd_, rd_, fd_, _ = prf(tpd, fpd, fnd, dc[("LOLOS", "LOLOS")])
    print(f"\n===== {name} =====")
    print(f"Baris tak terparse: {bad_rows}/{len(rs)} | butir tak terbaca: {missing}/{total}")
    print(f"TINGKAT DOKUMEN ({ndoc} TOR): akurasi {okdoc/ndoc:.1%} | precision REVISI {pd_:.1%} | recall REVISI {rd_:.1%} | F1 {fd_:.1%}")
    print(f"   gold LOLOS -> pred LOLOS {dc[('LOLOS','LOLOS')]}, pred REVISI {dc[('LOLOS','REVISI')]}")
    print(f"   gold REVISI -> pred REVISI {dc[('REVISI','REVISI')]}, pred LOLOS {dc[('REVISI','LOLOS')]}")
    print("\nPER BUTIR (kelas positif = TIDAK logis):"); display(df.round(3))
    print("PER METODE (ASLI/INDUK = unsur benar; lainnya = jenis kerusakan):"); display(dm.round(3))
    return df, dm

test_rows = rows["test"]

In [13]:
# --- model hasil fine-tune
if SKIP_TRAINING or "trainer" not in dir():
    model = PeftModel.from_pretrained(load_base(), ADAPTER_DIR)       # muat adapter dari ADAPTER_DIR (lihat sel 2)
else:
    model = trainer.model
try:
    model = model.merge_and_unload()                  # gabung LoRA -> inferensi lebih cepat
except Exception as e:
    print("merge dilewati:", e)
preds_ft = predict(model, test_rows, "FT")
df_ft, dm_ft = evaluate(test_rows, preds_ft, "Phi-4-mini + LoRA (fine-tune)")

with open(f"{OUT_DIR}/test_predictions.jsonl", "w", encoding="utf-8") as f:
    for r, p in zip(test_rows, preds_ft):
        pp = {code: get_logis(p["pred"], code) for code in r["_meta"]["labels"]}
        f.write(json.dumps({**r["_meta"], "pred_logis": pp, "raw": p["raw"]}, ensure_ascii=False) + "\n")
df_ft.to_csv(f"{OUT_DIR}/test_metrics_per_butir.csv", index=False)
dm_ft.to_csv(f"{OUT_DIR}/test_metrics_per_metode.csv", index=False)
print("prediksi & metrik disimpan ke", OUT_DIR)

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

model dimuat (percobaan 1): attn=sdpa, trust_remote_code=False


... (2 outputs left out)

,butir,n,tidak_logis_gold,akurasi,balanced_acc,precision_TIDAK,recall_TIDAK,F1_TIDAK
0,Dasar Hukum,56,27,1.000,1.000,1.000,1.000,1.000
1,Gambaran Umum,56,27,0.839,0.833,1.000,0.667,0.800
2,Latar Belakang,56,36,0.946,0.958,1.000,0.917,0.957
3,Metode Pelaksanaan,88,28,0.875,0.889,0.743,0.929,0.825
4,Tahapan dan Waktu Pelaksanaan,88,28,0.955,0.967,0.875,1.000,0.933
5,Strategi Pencapaian Keluaran,88,47,0.898,0.893,0.865,0.957,0.909
6,Penerima Manfaat,196,54,0.995,0.991,1.000,0.981,0.991
7,Kurun Waktu Pencapaian Keluaran,164,53,0.982,0.986,0.946,1.000,0.972
8,Biaya yang Diperlukan,216,53,0.977,0.972,0.944,0.962,0.953
9,Manajemen Risiko,208,53,0.981,0.987,0.930,1.000,0.964


PER METODE (ASLI/INDUK = unsur benar; lainnya = jenis kerusakan):


,metode,n,benar
0,ASLI,749,0.969
1,HAPUS_METODE,11,0.909
2,HAPUS_NOMINAL,25,0.920
3,HAPUS_REGULASI,10,1.000
4,HAPUS_SUBBAGIAN,14,0.929
5,HAPUS_VOLUME,8,0.125
6,HAPUS_WAKTU,34,1.000
7,INDUK,144,0.917
8,KOSONG,56,0.982
9,KOSONG_TABEL,25,1.000


prediksi & metrik disimpan ke /kaggle/working/phi4mini-perbutir-lora


In [14]:
# --- contoh kesalahan (untuk analisis)
shown = Counter()
for r, p in zip(test_rows, preds_ft):
    m = r["_meta"]
    for code, gold in m["labels"].items():
        pl = get_logis(p["pred"], code)
        if pl is None:
            pl = True
        if pl != gold and shown[gold] < 3:
            shown[gold] += 1
            print("=" * 90)
            print(f"{BUTIR_NAME.get(code, code)} | gold logis={gold} ({m['metode'][code]}) | prediksi logis={pl} | {m['tor_asal'][:60]}")
            uc = r["prompt"][1]["content"]
            print("ISI:", uc.split("\nBagian: ", 1)[-1][:600])
            print("JAWABAN MODEL:", p["raw"][:500])

Metode Pelaksanaan | gold logis=True (ASLI) | prediksi logis=False | TOR Layanan Hubungan Masyarakat dan Informasi 2027
ISI: Strategi Pencapaian Keluaran
A. Metode Pelaksanaan
Metode pelaksanaan Layanan Hubungan Masyarakat dan Informasi ini dilaksanakan oleh Tim Komunikasi Komisi Informasi Pusat dan penyedia jasa lainnya yang kompeten di bidang jurnalistik. Lokasi kegiatan dilaksanakan seluruhnya di lingkup Komisi Informasi Pusat. Tahap Pelaksanaan setiap Kegiatan Humas meliputi:
a. Tahap persiapan pemilihan konten b. Tahap koordinasi dengan stakeholder internal dan/atau eksternal c. Tahap penugasan dan pelaksanaan kepada penyedia dan staf d. Tahap evaluasi

B. Tahapan dan Waktu Pelaksanaan
xxx
JAWABAN MODEL: {"I.O.2": {"logis": false, "alasan": "Bagian ini menyatakan 'dilaksanakan oleh Tim Komunikasi Komisi Informasi Pusat dan penyedia jasa lainnya yang kompeten', sehingga cara pelaksanaan kontraktual melalui penyedia jasa. Namun, tidak ada keterangan bahwa pelaksanaan dilakukan secar

In [15]:
# --- (opsional) pembanding: model dasar tanpa fine-tune
if EVAL_BASELINE:
    # trainer (dan optimizer-nya) masih memegang referensi ke model fine-tune di GPU;
    # del model saja tidak cukup -> OOM saat load_base() dipanggil lagi. Harus del trainer juga.
    try: del trainer
    except NameError: pass
    try: del model
    except NameError: pass
    gc.collect(); torch.cuda.empty_cache()
    print(f"GPU terpakai sebelum muat base: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    base = load_base()
    preds_base = predict(base, test_rows, "BASE")
    df_base, dm_base = evaluate(test_rows, preds_base, "Phi-4-mini-instruct (tanpa fine-tune)")
    df_base.to_csv(f"{OUT_DIR}/test_metrics_per_butir_BASELINE.csv", index=False)
    del base; gc.collect(); torch.cuda.empty_cache()

GPU terpakai sebelum muat base: 0.0 GB


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

model dimuat (percobaan 1): attn=sdpa, trust_remote_code=False


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3280: FutureWarning: `gradient_checkpointing_disable` is deprecated and will be removed in v6. Use `activation_checkpointing_disable` instead.
  warnings.warn(


BASE 8/928  0.9 menit, sisa ~98.2 menit
BASE 88/928  8.1 menit, sisa ~77.0 menit
BASE 168/928  14.1 menit, sisa ~63.6 menit
BASE 248/928  18.4 menit, sisa ~50.6 menit
BASE 328/928  21.1 menit, sisa ~38.5 menit
BASE 408/928  22.8 menit, sisa ~29.1 menit
BASE 488/928  24.2 menit, sisa ~21.8 menit
BASE 568/928  25.5 menit, sisa ~16.1 menit
BASE 648/928  26.7 menit, sisa ~11.5 menit
BASE 728/928  27.9 menit, sisa ~7.7 menit
BASE 808/928  29.0 menit, sisa ~4.3 menit
BASE 888/928  29.9 menit, sisa ~1.3 menit
BASE 928/928  30.4 menit, sisa ~0.0 menit

===== Phi-4-mini-instruct (tanpa fine-tune) =====
Baris tak terparse: 38/928 | butir tak terbaca: 1215/1216
TINGKAT DOKUMEN (228 TOR): akurasi 25.0% | precision REVISI nan% | recall REVISI 0.0% | F1 nan%
   gold LOLOS -> pred LOLOS 57, pred REVISI 0
   gold REVISI -> pred REVISI 0, pred LOLOS 171

PER BUTIR (kelas positif = TIDAK logis):


,butir,n,tidak_logis_gold,akurasi,balanced_acc,precision_TIDAK,recall_TIDAK,F1_TIDAK
0,Dasar Hukum,56,27,0.518,0.5,NaN,0.0,NaN
1,Gambaran Umum,56,27,0.518,0.5,NaN,0.0,NaN
2,Latar Belakang,56,36,0.357,0.5,NaN,0.0,NaN
3,Metode Pelaksanaan,88,28,0.682,0.5,NaN,0.0,NaN
4,Tahapan dan Waktu Pelaksanaan,88,28,0.682,0.5,NaN,0.0,NaN
5,Strategi Pencapaian Keluaran,88,47,0.466,0.5,NaN,0.0,NaN
6,Penerima Manfaat,196,54,0.724,0.5,NaN,0.0,NaN
7,Kurun Waktu Pencapaian Keluaran,164,53,0.677,0.5,NaN,0.0,NaN
8,Biaya yang Diperlukan,216,53,0.755,0.5,NaN,0.0,NaN
9,Manajemen Risiko,208,53,0.745,0.5,NaN,0.0,NaN


PER METODE (ASLI/INDUK = unsur benar; lainnya = jenis kerusakan):


,metode,n,benar
0,ASLI,749,1.000
1,HAPUS_METODE,11,0.000
2,HAPUS_NOMINAL,25,0.000
3,HAPUS_REGULASI,10,0.000
4,HAPUS_SUBBAGIAN,14,0.000
5,HAPUS_VOLUME,8,0.000
6,HAPUS_WAKTU,34,0.000
7,INDUK,144,0.424
8,KOSONG,56,0.000
9,KOSONG_TABEL,25,0.000


## 10. (Tambahan) Evaluasi kualitas *reasoning* — perplexity & BERTScore
Usulan Mbak Fira di weekly meeting: metrik di section 9 hanya menilai **label** (logis/tidak logis), belum menilai **kualitas teks alasannya**. Dua metrik tambahan:
- **Perplexity**: seberapa "wajar" teks alasan **gold** di bawah distribusi model hasil fine-tune (teacher forcing — token prompt di-mask, sama seperti perhitungan loss saat training). Dihitung dari forward pass saja (tanpa generate) sehingga jauh lebih murah daripada evaluasi di section 9; dipecah per kelompok butir dan per status dokumen (LOLOS vs REVISI). Sebagai pembanding kasar: `eval_loss` akhir saat training (section 8) ±0.39 setara perplexity ±exp(0.39)≈1.48 — angka di sini seharusnya berada di kisaran yang sama.
- **BERTScore**: kemiripan semantik antara alasan yang **digenerate model** (dipakai ulang dari prediksi section 9) dengan alasan **gold**, per butir — dipecah pula antara prediksi yang **label-nya benar** vs **salah**, untuk mengecek apakah alasannya tetap masuk akal walau verdict-nya keliru. Memakai model multibahasa (mBERT) karena teksnya Bahasa Indonesia; unduhan ±700 MB sekali jalan (butuh *Internet: On*, sama seperti kebutuhan mengunduh Qwen3-4B).

Kedua metrik ini melengkapi, bukan menggantikan, metrik per butir di section 9.

In [16]:
PPL_SUBSET = 500   # None = pakai seluruh test set (forward pass saja, jauh lebih murah daripada generate)

# model fine-tune mungkin sudah dihapus di sel EVAL_BASELINE (section 9) untuk hemat memori -> muat ulang
try:
    model
except NameError:
    model = PeftModel.from_pretrained(load_base(), ADAPTER_DIR)
    try:
        model = model.merge_and_unload()
    except Exception as e:
        print("merge dilewati:", e)

@torch.no_grad()
def completion_perplexity(model, rs):
    model.eval(); model.config.use_cache = False
    per_group, per_status = defaultdict(lambda: [0.0, 0]), defaultdict(lambda: [0.0, 0])
    total_nll = total_tok = 0
    for n, r in enumerate(rs, 1):
        full = tok.apply_chat_template(r["prompt"] + r["completion"], tokenize=False)
        prompt_only = tok.apply_chat_template(r["prompt"], tokenize=False, add_generation_prompt=True)
        enc = tok(full, return_tensors="pt", add_special_tokens=False).to(model.device)
        n_prompt = len(tok(prompt_only, add_special_tokens=False)["input_ids"])
        labels = enc["input_ids"].clone()
        labels[:, :n_prompt] = -100            # loss hanya di token completion, sama seperti saat training
        n_comp_tok = int((labels != -100).sum())
        if n_comp_tok == 0:
            continue
        nll = model(**enc, labels=labels).loss.item() * n_comp_tok
        total_nll += nll; total_tok += n_comp_tok
        g, st = r["_meta"]["group"], r["_meta"]["doc_status"]
        per_group[g][0] += nll; per_group[g][1] += n_comp_tok
        per_status[st][0] += nll; per_status[st][1] += n_comp_tok
        if n % 50 == 0 or n == len(rs):
            print(f"perplexity {n}/{len(rs)}")
    model.config.use_cache = True
    rows_out = [{"kelompok": "SEMUA", "n_token": total_tok, "perplexity": math.exp(total_nll / total_tok)}]
    rows_out += [{"kelompok": g, "n_token": t, "perplexity": math.exp(nll / t)} for g, (nll, t) in sorted(per_group.items())]
    rows_out += [{"kelompok": f"doc_status={s}", "n_token": t, "perplexity": math.exp(nll / t)} for s, (nll, t) in sorted(per_status.items())]
    return pd.DataFrame(rows_out)

ppl_rows = test_rows if PPL_SUBSET is None else rng.sample(test_rows, min(PPL_SUBSET, len(test_rows)))
df_ppl = completion_perplexity(model, ppl_rows)
print(f"perplexity dihitung dari {len(ppl_rows)}/{len(test_rows)} baris test")
display(df_ppl.round(3))
df_ppl.to_csv(f"{OUT_DIR}/test_perplexity.csv", index=False)

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

model dimuat (percobaan 1): attn=sdpa, trust_remote_code=False


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


perplexity 50/500
perplexity 100/500
perplexity 150/500
perplexity 200/500
perplexity 250/500
perplexity 300/500
perplexity 350/500
perplexity 400/500
perplexity 450/500
perplexity 500/500
perplexity dihitung dari 500/928 baris test


,kelompok,n_token,perplexity
0,SEMUA,62983,1.577
1,Biaya yang Diperlukan,10948,1.621
2,Kurun Waktu Pencapaian Keluaran,8633,1.583
3,Latar Belakang,9623,1.534
4,Manajemen Risiko,11011,1.565
5,Penerima Manfaat,9250,1.600
6,Strategi Pencapaian Keluaran,13518,1.565
7,doc_status=LOLOS,13543,1.582
8,doc_status=REVISI,49440,1.576


In [17]:
!pip install -q bert-score

In [18]:
from bert_score import score as bertscore

def extract_alasan(parsed, code):
    v = parsed.get(code) if isinstance(parsed, dict) else None
    return v.get("alasan") if isinstance(v, dict) and isinstance(v.get("alasan"), str) else None

try:
    preds_ft
except NameError:                                      # sesi baru: muat prediksi dari hasil section 9
    preds_ft = []
    with open(f"{OUT_DIR}/test_predictions.jsonl", encoding="utf-8") as f:
        for ln in f:
            d = json.loads(ln)
            preds_ft.append({"raw": d["raw"], "pred": parse_output(d["raw"])})

cands, refs, tag_butir, tag_benar = [], [], [], []
for r, p in zip(test_rows, preds_ft):
    gold = json.loads(r["completion"][0]["content"])
    for code, gold_logis in r["_meta"]["labels"].items():
        ref, cand = extract_alasan(gold, code), extract_alasan(p["pred"], code)
        if ref and cand:
            cands.append(cand); refs.append(ref)
            tag_butir.append(BUTIR_NAME.get(code, code))
            tag_benar.append(get_logis(p["pred"], code) == gold_logis)

print(f"pasangan alasan yang bisa dibandingkan: {len(cands)}/{sum(len(r['_meta']['labels']) for r in test_rows)}")
_, _, F1 = bertscore(cands, refs, model_type="bert-base-multilingual-cased", batch_size=32, verbose=True)

df_bs = pd.DataFrame({"butir": tag_butir, "prediksi_benar": tag_benar, "bertscore_f1": F1.tolist()})
print("\nBERTScore F1 rata-rata per butir:")
display(df_bs.groupby("butir")["bertscore_f1"].agg(["mean", "count"]).round(3))
print("\nBERTScore F1: prediksi benar vs salah (cek apakah alasan tetap masuk akal walau verdict keliru)")
display(df_bs.groupby("prediksi_benar")["bertscore_f1"].agg(["mean", "count"]).round(3))
df_bs.to_csv(f"{OUT_DIR}/test_bertscore.csv", index=False)

pasangan alasan yang bisa dibandingkan: 1216/1216


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

... (7 outputs left out)

done in 7.60 seconds, 159.97 sentences/sec

BERTScore F1 rata-rata per butir:


,mean,count
butir,,
Biaya yang Diperlukan,0.835,216
Dasar Hukum,0.838,56
Gambaran Umum,0.812,56
Kurun Waktu Pencapaian Keluaran,0.838,164
Latar Belakang,0.816,56
Manajemen Risiko,0.833,208
Metode Pelaksanaan,0.843,88
Penerima Manfaat,0.837,196
Strategi Pencapaian Keluaran,0.822,88



BERTScore F1: prediksi benar vs salah (cek apakah alasan tetap masuk akal walau verdict keliru)


,mean,count
prediksi_benar,,
False,0.768,49
True,0.834,1167


## 11. (Opsional) Simpan model gabungan untuk deploy
Model fp16 utuh (~8 GB) yang bisa dijalankan langsung dengan vLLM/TGI tanpa PEFT. Hasilnya ada di `/kaggle/working` -> unduh atau *Save Version* untuk menyimpannya sebagai output.

In [19]:
MERGE_SAVE = False
if MERGE_SAVE:
    try: del model
    except NameError: pass
    gc.collect(); torch.cuda.empty_cache()
    merged = PeftModel.from_pretrained(load_base(), f"{OUT_DIR}/final_adapter").merge_and_unload()
    merged.config.use_cache = True
    merged.save_pretrained(f"{OUT_DIR}/merged_fp16", safe_serialization=True)
    tok.save_pretrained(f"{OUT_DIR}/merged_fp16")
    print("model gabungan disimpan ke", f"{OUT_DIR}/merged_fp16")